# ESM-2 model usage with `transformers`

notebook based on: https://github.com/huggingface/notebooks/blob/main/examples/protein_language_modeling.ipynb
new elements:
- sequence classification by PEFT
- sequence classification with KNN using ESM-2 as feature extractor
- sequence classification with custom model

**Rodzina modeli ESM-2**

ESM-2 występuje w kilku rozmiarach modelu bazowego, od 8 mln do 8 mld parametrów. Tutaj będziemy wykorzystywać mały model 35M https://huggingface.co/facebook/esm2_t12_35M_UR50D

Wersja modelu bazowego jest określona w zmiennej `model_checkpoint = "facebook/esm2_t12_35M_UR50D"`

Hugging Face oferuje różne architektury zadaniowe (task-specific architectures) dla danego modelu bazowego, z których każda wykorzystuje inną głowę/głowicę modelu (head) do konkretnych zadań końcowych

## 1. Set-up, check if works
install, download, initialize tokenizer and model, check what is the output for a given sequence

In [1]:
!uv pip install evaluate peft==0.18

Using Python 3.12.13 environment at: /usr
Resolved 79 packages in 2.33s
Prepared 2 packages in 167ms
Uninstalled 1 package in 57ms
Installed 2 packages in 29ms
 + evaluate==0.4.6
 - peft==0.19.1
 + peft==0.18.0


In [2]:
from transformers import AutoTokenizer, EsmModel
import torch

model_checkpoint = "facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = EsmModel.from_pretrained(model_checkpoint)

sequence = "MAKWGEGDPRWIVEERADATNVNNWHWTERDASNWSTDKLKTLFLAVQVQNEEGKCEVTEVSKLDGEASINNRKGKLIFFYEWSV"
inputs = tokenizer(sequence, return_tensors="pt")
outputs = model(**inputs)
print("Hidden state shape:", outputs.last_hidden_state.shape, "(batch_size, sequence_length, hidden_size)")
print("Pooled output shape:", outputs.pooler_output.shape, "(batch_size, hidden_size)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Hidden state shape: torch.Size([1, 87, 480]) (batch_size, sequence_length, hidden_size)
Pooled output shape: torch.Size([1, 480]) (batch_size, hidden_size)


In [3]:
# Co zwraca tokenizer?
print(inputs)

for key, value in inputs.items():
    print(f"{key}: {value.shape} (batch size: {value.shape[0]}, sequence length: {value.shape[1]})")

{'input_ids': tensor([[ 0, 20,  5, 15, 22,  6,  9,  6, 13, 14, 10, 22, 12,  7,  9,  9, 10,  5,
         13,  5, 11, 17,  7, 17, 17, 22, 21, 22, 11,  9, 10, 13,  5,  8, 17, 22,
          8, 11, 13, 15,  4, 15, 11,  4, 18,  4,  5,  7, 16,  7, 16, 17,  9,  9,
          6, 15, 23,  9,  7, 11,  9,  7,  8, 15,  4, 13,  6,  9,  5,  8, 12, 17,
         17, 10, 15,  6, 15,  4, 12, 18, 18, 19,  9, 22,  8,  7,  2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
input_ids: torch.Size([1, 87]) (batch size: 1, sequence length: 87)
attention_mask: torch.Size([1, 87]) (batch size: 1, sequence length: 87)


`input_ids` to indeksy tokenów, `attention_mask` zawiera 1, jeśli dany token jest ważny i 0, jeśli należy go pominąć (potrzebne np w paddingu)

In [4]:
print("Długość sekwencji:", len(sequence))
print("Długość tokenów:", len(tokenizer.encode(sequence)))
print(tokenizer.decode(inputs["input_ids"][0]))
print("Wielkość słownika:", tokenizer.vocab_size)

Długość sekwencji: 85
Długość tokenów: 87
<cls> M A K W G E G D P R W I V E E R A D A T N V N N W H W T E R D A S N W S T D K L K T L F L A V Q V Q N E E G K C E V T E V S K L D G E A S I N N R K G K L I F F Y E W S V <eos>
Wielkość słownika: 33


In [5]:
# Get the full vocabulary dictionary
vocab = tokenizer.get_vocab()
for key, value in vocab.items():
    print(f"{key}: {value}")

<cls>: 0
<pad>: 1
<eos>: 2
<unk>: 3
L: 4
A: 5
G: 6
V: 7
S: 8
E: 9
R: 10
T: 11
I: 12
D: 13
P: 14
K: 15
Q: 16
N: 17
F: 18
Y: 19
M: 20
H: 21
W: 22
C: 23
X: 24
B: 25
U: 26
Z: 27
O: 28
.: 29
-: 30
<null_1>: 31
<mask>: 32


- `.` and `-` are used in protein sequencies
- `<eos>` is the special token used to indicate the end of a sequence
- `<unk>` is the special token used for unknown or out-of-vocabulary tokens that are not present in the tokenizer's vocabulary
- `<mask>` is the special token used for masked language modeling tasks, where certain tokens in the input sequence are replaced with <mask> and the model is trained to predict the original token
- `<null_1>` is an unused special token that may be reserved for future use or specific tasks, but it does not have a predefined meaning in the context of the ESM model

## 2. Run as is
mask selected amino-acids and check the output probabilities

In [ ]:
# token maski
print("Token maski i jego ID:", tokenizer.mask_token, tokenizer.mask_token_id)

print("Dekodowany token maski:", tokenizer.decode(tokenizer.mask_token_id))

print("Kodowany token maski:", tokenizer.encode("<mask>"), "(początek sekwencji, token maski, koniec sekwencji)")

In [6]:
from transformers import EsmForMaskedLM

masked_lm = EsmForMaskedLM.from_pretrained(model_checkpoint)
sequence_with_mask = "MAKWGEGDPRWIVEERADATNVNNWHWTERDASNWSTDKLKTLFLAVQVQNEEGKCEVTEVSKLDGEASINNRKGKLIFFYEWS<mask>"
inputs = tokenizer(sequence_with_mask, return_tensors="pt")


with torch.no_grad():
    outputs = masked_lm(**inputs)
logits = outputs.logits

print("Logits shape:", logits.shape, "(batch size, sequence length, vocab size)")

mask_token_indices = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
predicted_tokens_id = logits[0, mask_token_indices].argmax(axis=-1)
for token_index, predicted_token_id in zip(mask_token_indices, predicted_tokens_id):
    print(f"Predicted token ID for position {token_index.item()-1}: {predicted_token_id.item()}")


Loading weights:   0%|          | 0/214 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Logits shape: torch.Size([1, 87, 33]) (batch_size, sequence_length, vocab_size)
Predicted amino acid: L


## 3. Fine-tune for sequence classification
as in `protein_language_modeling.ipynb`

In [7]:
import requests
from io import BytesIO
import pandas as pd

query_url ="https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Csequence%2Ccc_subcellular_location&format=tsv&query=%28%28organism_id%3A9606%29%20AND%20%28reviewed%3Atrue%29%20AND%20%28length%3A%5B80%20TO%20500%5D%29%29"
uniprot_request = requests.get(query_url)
bio = BytesIO(uniprot_request.content)
df = pd.read_csv(bio, compression='gzip', sep='\t')

df = df.dropna()
cytosolic = df['Subcellular location [CC]'].str.contains("Cytosol") | df['Subcellular location [CC]'].str.contains("Cytoplasm")
membrane = df['Subcellular location [CC]'].str.contains("Membrane") | df['Subcellular location [CC]'].str.contains("Cell membrane")

cytosolic_df = df[cytosolic & ~membrane]
membrane_df = df[membrane & ~cytosolic]

cytosolic_sequences = cytosolic_df["Sequence"].tolist()
cytosolic_labels = [0 for _ in cytosolic_sequences]
membrane_sequences = membrane_df["Sequence"].tolist()
membrane_labels = [1 for _ in membrane_sequences]

sequences = cytosolic_sequences + membrane_sequences
labels = cytosolic_labels + membrane_labels

print("Liczba sekwencji cytosolowych:", len(cytosolic_sequences))
print("Liczba sekwencji membranowych:", len(membrane_sequences))

In [ ]:
print("przykładowa sekwencja:", sequences[0])
print("przykładowa etykieta:", labels[0])

In [8]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

train_sequences, test_sequences, train_labels, test_labels = train_test_split(sequences, labels, test_size=0.25, shuffle=True)
train_tokenized = tokenizer(train_sequences)
test_tokenized = tokenizer(test_sequences)

In [9]:
from datasets import Dataset

train_dataset = Dataset.from_dict(train_tokenized).add_column("labels", train_labels)
test_dataset = Dataset.from_dict(test_tokenized).add_column("labels", test_labels)

In [ ]:
print("Dataset zwraca słownik z kluczami:", train_dataset.column_names)
print("Przykładowy element datasetu:", train_dataset[0])


Potrzebujemy zrobić konwersję modelu z przeiwdywania zasłoniętego tokenu na klasyfikację. Z `transformers` można to zrobić jednym poleceniem `AutoModelForSequenceClassification.from_pretrained`

In [10]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

num_labels = 2
clf_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

args = TrainingArguments(
    "esm2-finetuned-localization",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    clf_model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)



Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2}.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.191811,0.944615


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.LayerNorm.bias', 'esm.encoder.layer.1.LayerNorm.weight', 'esm.encoder.layer.1.LayerNorm.bias', 'esm.encoder.layer.2.attention.LayerNorm.weight', 'esm.encoder.layer.2.attention.LayerNorm.bias', 'esm.encoder.layer.2.LayerNorm.weight', 'esm.encoder.layer.2.LayerNorm.bias', 'esm.encoder.layer.3.attention.LayerNorm.weight', 'esm.encoder.layer.3.attention.LayerNorm.bias', 'esm.encoder.layer.3.LayerNorm.weight', 'esm.encoder.layer.3.LayerNorm.bias', 'esm.encoder.layer.4.attention.LayerNorm.weight', 'esm.encoder.layer.4.attention.LayerNorm.bias', 'esm.encoder.layer.4.LayerNorm.weight', 'esm.encoder.layer.4.LayerNorm.bias', 'esm.encoder.layer.5.attention.LayerNorm.weight', 'esm.encoder.

TrainOutput(global_step=488, training_loss=0.2568389079609855, metrics={'train_runtime': 240.4415, 'train_samples_per_second': 16.22, 'train_steps_per_second': 2.03, 'total_flos': 350864256720024.0, 'train_loss': 0.2568389079609855, 'epoch': 1.0})

## 4. PEFT for sequence classification
as before, but with PEFT

In [40]:
print(clf_model)

EsmForSequenceClassification(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 480, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-11): 12 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=480, out_features=480, bias=True)
              (key): Linear(in_features=480, out_features=480, bias=True)
              (value): Linear(in_features=480, out_features=480, bias=True)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=480, out_features=480, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((480,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_featur

Jak wybrać warstwy do modyfikacji?

Zwykle wszystkie `query` i `value`, czasami również `key` i `dense`

In [12]:
from peft import get_peft_model, LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # Sequence Classification
    inference_mode=False, # False for training, True for inference
    r=8, # rank
    lora_alpha=16, # Scaling Factor. Low rank adapter is multiplied by alpha/r and added to weights
    lora_dropout=0.1,
    target_modules=["query", "value"] # ESM attention modules
)

peft_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)
peft_model = get_peft_model(peft_model, peft_config)
peft_model.print_trainable_parameters()

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 416,162 || all params: 33,917,525 || trainable%: 1.2270


In [13]:
peft_args = TrainingArguments(
    "esm2-peft-localization",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

peft_trainer = Trainer(
    peft_model,
    peft_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

peft_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2}.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.160176,0.950769


TrainOutput(global_step=488, training_loss=0.22234500822473746, metrics={'train_runtime': 207.94, 'train_samples_per_second': 18.755, 'train_steps_per_second': 2.347, 'total_flos': 355224838778280.0, 'train_loss': 0.22234500822473746, 'epoch': 1.0})

## 5. Fine tune for token classification


In [14]:
# Secondary structure prediction (Token classification)
query_url ="https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Csequence%2Cft_helix%2Cft_strand%2Cft_turn&format=tsv&query=%28%28organism_id%3A9606%29%20AND%20%28reviewed%3Atrue%29%20AND%20%28length%3A%5B80%20TO%20500%5D%29%29"
uniprot_request = requests.get(query_url)
bio = BytesIO(uniprot_request.content)
df = pd.read_csv(bio, compression='gzip', sep='\t')

In [19]:
no_structure_rows = df["Beta strand"].isna() & df["Helix"].isna()
df = df[~no_structure_rows]
df

,Entry,Sequence,Helix,Beta strand,Turn
2,A0A2R8Y7D0,MEGLRRGLSRWKRYHIKVHLADEALLLPLTVRPRDTLSDLRAQLVG...,"HELIX 4..7; /evidence=""ECO:0007829|PDB:7MRJ""; ...","STRAND 14..20; /evidence=""ECO:0007829|PDB:7MRJ...","TURN 8..11; /evidence=""ECO:0007829|PDB:7MRJ""; ..."
5,A0JLT2,MENFTALFGAQADPPPPPTALGFGPGKPPPPPPPPAGGGPGTAPPP...,"HELIX 83..86; /evidence=""ECO:0007829|PDB:7EMF""...","STRAND 79..81; /evidence=""ECO:0007829|PDB:7EMF""","TURN 117..119; /evidence=""ECO:0007829|PDB:7EMF"""
17,A1L3X0,MAFSDLTSRTVHLYDNWIKDADPRVEDWLLMSSPLPQTILLGFYVY...,"HELIX 17..20; /evidence=""ECO:0007829|PDB:6Y7F""...","STRAND 97..99; /evidence=""ECO:0007829|PDB:6Y7F""","TURN 89..94; /evidence=""ECO:0007829|PDB:6Y7F"""
18,A1XBS5,MMRRTLENRNAQTKQLQTAVSNVEKHFGELCQIFAAYVRKTARLRD...,"HELIX 2..6; /evidence=""ECO:0007829|PDB:8CEG""; ...",NaN,NaN
19,A1Z1Q3,MYPSNKKKKVWREEKERLLKMTLEERRKEYLRDYIPLNSILSWKEE...,"HELIX 11..19; /evidence=""ECO:0007829|PDB:4IQY""...","STRAND 71..77; /evidence=""ECO:0007829|PDB:4IQY...","TURN 27..29; /evidence=""ECO:0007829|PDB:4IQY"";..."
...,...,...,...,...,...
11604,Q96I45,MVNLGLSRVDDAVAAKHPGLGEYAACQSHAFMKGVFTFVTGTGMAF...,"HELIX 6..16; /evidence=""ECO:0007829|PDB:2LOR"";...","STRAND 3..5; /evidence=""ECO:0007829|PDB:2LOR"";...",NaN
11658,Q9H0W7,MPTNCAAAGCATTYNKHINISFHRFPLDPKRRKEWVRLVRRKNFVP...,"HELIX 29..38; /evidence=""ECO:0007829|PDB:2D8R""","STRAND 7..9; /evidence=""ECO:0007829|PDB:2D8R"";...",NaN
11695,Q9P1F3,MNVDHEVNLLVEEIHRLGSKNADGKLSVKFGVLFRDDKCANLFEAL...,"HELIX 3..17; /evidence=""ECO:0007829|PDB:2L2O"";...","STRAND 24..29; /evidence=""ECO:0007829|PDB:2L2O...",NaN
11697,Q9P298,MSANRRWWVPPDDEDCVSEKLLRKTRESPLVPIGLGGCLVVAAYRI...,"HELIX 18..24; /evidence=""ECO:0007829|PDB:2LON""...","STRAND 11..14; /evidence=""ECO:0007829|PDB:2LON...",NaN


In [41]:
import re
import numpy as np

def get_positions(x):
    positions = []
    if pd.isna(x):
        return positions
    # Modified regex: ensure capturing groups for numbers are always present
    for match in re.finditer(r"(?:STRAND|HELIX|TURN)\s+(\d+)\.\.(\d+)", x):
        try:
            start, end = int(match.group(1)), int(match.group(2))
            positions.append((start, end))
        except ValueError:
            # This except block will now only catch cases where the captured groups
            # are not valid integers, which is less likely with the corrected regex.
            pass
    return positions

df["helix_pos"] = df["Helix"].apply(get_positions)
df["strand_pos"] = df["Beta strand"].apply(get_positions)
df["turn_pos"] = df["Turn"].apply(get_positions)

def generate_labels(row):
    seq_len = len(row["Sequence"])
    labels = np.zeros(seq_len, dtype=int)

    # 0 - uknown, 1 - helix, 2 - strand, 3- turn
    for start, end in row["helix_pos"]:
        if start - 1 < seq_len and end <= seq_len:
            labels[start-1:end] = 1
    for start, end in row["strand_pos"]:
        if start - 1 < seq_len and end <= seq_len:
            labels[start-1:end] = 2
    for start, end in row["turn_pos"]:
        if start - 1 < seq_len and end <= seq_len:
            labels[start-1:end] = 3
    return labels.tolist()

df["labels"] = df.apply(generate_labels, axis=1)

token_sequences = df["Sequence"].tolist()
token_labels = df["labels"].tolist()

In [42]:
print(f"Przykładowa sekwencja ({len(token_sequences[0])}):", token_sequences[0])
print(f"Przykładowe etykiety ({len(token_labels[0])}):", token_labels[0])

Przykładowa sekwencja (87): MEGLRRGLSRWKRYHIKVHLADEALLLPLTVRPRDTLSDLRAQLVGQGVSSWKRAFYYNARRLDDHQTVRDARLQDGSVLLLVSDPR
Przykładowe etykiety (87): [0, 0, 0, 1, 1, 1, 1, 3, 3, 3, 3, 0, 0, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 0, 0, 0]


In [43]:
train_token_seqs, test_token_seqs, train_token_labs, test_token_labs = train_test_split(
    token_sequences, token_labels, test_size=0.25, shuffle=True
)

train_token_tokenized = tokenizer(train_token_seqs)
test_token_tokenized = tokenizer(test_token_seqs)

# Pad labels to match sequence lengths (accounting for special tokens [CLS] and [EOS])
def align_labels(tokenized, labels):
    aligned_labels = []
    for i in range(len(labels)):
        seq_labels = labels[i]
        input_ids = tokenized["input_ids"][i]

        # -100 is the ignore index for PyTorch CrossEntropyLoss
        aligned = [-100] + seq_labels + [-100]
        aligned_labels.append(aligned)
    return aligned_labels

train_token_tokenized["labels"] = align_labels(train_token_tokenized, train_token_labs)
test_token_tokenized["labels"] = align_labels(test_token_tokenized, test_token_labs)

train_token_dataset = Dataset.from_dict(train_token_tokenized)
test_token_dataset = Dataset.from_dict(test_token_tokenized)

In [47]:
from transformers import AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification


# 0: Coil, 1: Helix, 2: Strand, 3: Turn
token_model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=4)

token_args = TrainingArguments(
    "esm2-finetuned-secondary-structure",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

)

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy_metric(eval_pred):
    predictions, labels = eval_pred
    labels = labels.reshape((-1,))
    predictions = np.argmax(predictions, axis=2)
    predictions = predictions.reshape((-1,))
    predictions = predictions[labels!=-100] # -100 is the ignore index for PyTorch CrossEntropyLoss
    labels = labels[labels!=-100] # as above
    return accuracy_metric.compute(predictions=predictions, references=labels)

data_collator = DataCollatorForTokenClassification(tokenizer)


token_trainer = Trainer(
    token_model,
    token_args,
    train_dataset=train_token_dataset,
    eval_dataset=test_token_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_accuracy_metric
,
)

token_trainer.train()

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmForTokenClassification LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.648924,0.759881


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['esm.encoder.layer.0.attention.LayerNorm.weight', 'esm.encoder.layer.0.attention.LayerNorm.bias', 'esm.encoder.layer.0.LayerNorm.weight', 'esm.encoder.layer.0.LayerNorm.bias', 'esm.encoder.layer.1.attention.LayerNorm.weight', 'esm.encoder.layer.1.attention.LayerNorm.bias', 'esm.encoder.layer.1.LayerNorm.weight', 'esm.encoder.layer.1.LayerNorm.bias', 'esm.encoder.layer.2.attention.LayerNorm.weight', 'esm.encoder.layer.2.attention.LayerNorm.bias', 'esm.encoder.layer.2.LayerNorm.weight', 'esm.encoder.layer.2.LayerNorm.bias', 'esm.encoder.layer.3.attention.LayerNorm.weight', 'esm.encoder.layer.3.attention.LayerNorm.bias', 'esm.encoder.layer.3.LayerNorm.weight', 'esm.encoder.layer.3.LayerNorm.bias', 'esm.encoder.layer.4.attention.LayerNorm.weight', 'esm.encoder.layer.4.attention.LayerNorm.bias', 'esm.encoder.layer.4.LayerNorm.weight', 'esm.encoder.layer.4.LayerNorm.bias', 'esm.encoder.layer.5.attention.LayerNorm.weight', 'esm.encoder.

TrainOutput(global_step=412, training_loss=0.7729808474049985, metrics={'train_runtime': 204.6885, 'train_samples_per_second': 16.093, 'train_steps_per_second': 2.013, 'total_flos': 298367425746120.0, 'train_loss': 0.7729808474049985, 'epoch': 1.0})

## 6. Protein localization classification using KNN on ESM-2 embeddings
Using the pre-trained ESM-2 model to extract sequence embeddings and training a K-Nearest Neighbors classifier.

In [36]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import torch
from tqdm.auto import tqdm
import numpy as np

# We will use the base ESM-2 model loaded in section 1 and the localization dataset from section 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

def get_sequence_embeddings(seqs, batch_size=8):
    embeddings = []
    for i in tqdm(range(0, len(seqs), batch_size)):
        batch_seqs = seqs[i:i+batch_size]
        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            # Use the mean of the hidden states (excluding padding tokens)
            attention_mask = inputs['attention_mask'].unsqueeze(-1)
            sum_embeddings = torch.sum(outputs.last_hidden_state * attention_mask, dim=1)
            sum_mask = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
            mean_embeddings = sum_embeddings / sum_mask
            embeddings.append(mean_embeddings.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

# Extract embeddings
print("Extracting training embeddings...")
X_train = get_sequence_embeddings(train_sequences)
print("Extracting testing embeddings...")
X_test = get_sequence_embeddings(test_sequences)

# Train KNN
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(X_train, train_labels)

# Predict and evaluate
y_pred = knn.predict(X_test)
accuracy = accuracy_score(test_labels, y_pred)
print(f"KNN Accuracy on ESM-2 embeddings: {accuracy:.4f}")

Extracting training embeddings...


  0%|          | 0/488 [00:00<?, ?it/s]

Extracting testing embeddings...


  0%|          | 0/163 [00:00<?, ?it/s]

KNN Accuracy on ESM-2 embeddings: 0.9315


In [37]:
# Train KNN
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(X_train, train_labels)

# Predict and evaluate
y_pred = knn.predict(X_test)
accuracy = accuracy_score(test_labels, y_pred)
print(f"KNN Accuracy on ESM-2 embeddings: {accuracy:.4f}")

KNN Accuracy on ESM-2 embeddings: 0.9315


## 7. Custom classifier

Ponownie klasyfikacja lokalizacji białka, tym razem z użyciem własnego modelu

In [38]:
from torch import nn

num_labels = 2 # 0: Cytosolic, 1: Membrane

class SimpleEsmClassifier(nn.Module):
    def __init__(self, backbone, num_labels, dropout=0.2, freeze_backbone=False):
        super().__init__()
        self.num_labels = num_labels
        self.backbone = backbone
        hidden_size = self.backbone.config.hidden_size

        # Optionally freeze the backbone parameters, reducing memory usage and speeding up training
        # However, it would be better to have larger self.classifier capacity in this case
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels) # you may change it to multi layer network e.g. nn.Sequential(nn.Linear(hidden_size, hidden_size//2), nn.GELU(), nn.Linear(hidden_size//2, num_labels))
        self.loss_fn = nn.CrossEntropyLoss() # standard loss for multi-class classification

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # Sequence-level representation
        pooled = outputs.pooler_output # shape: (batch_size, hidden_size)
        logits = self.classifier(self.dropout(pooled)) # shape: (batch_size, num_labels)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

backbone = EsmModel.from_pretrained(model_checkpoint)

custom_classifier = SimpleEsmClassifier(backbone, num_labels, freeze_backbone=False)

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
args = TrainingArguments(
    "esm2-finetuned-localization-custom",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    custom_classifier,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.180504,0.943846


TrainOutput(global_step=488, training_loss=0.2426720603567655, metrics={'train_runtime': 237.7927, 'train_samples_per_second': 16.401, 'train_steps_per_second': 2.052, 'total_flos': 0.0, 'train_loss': 0.2426720603567655, 'epoch': 1.0})